In [4]:
pip install openmeteo-requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.1/167.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.0/684.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.8/145.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.1 MB/s eta 0:00:00


In [5]:
pip install requests-cache retry-requests numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.2 MB/s eta 0:00:00


In [20]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

locations = [
    {"name": "Bandung", "latitude": -6.9222, "longitude": 107.6069},
    {"name": "Jakarta", "latitude": -6.2088, "longitude": 106.8456},
    {"name": "Surabaya", "latitude": -7.2575, "longitude": 112.7521},
    {"name": "Bali", "latitude": -8.3333, "longitude": 115},
    {"name": "Pekanbaru", "latitude": 0.5167, "longitude": 101.4417},
    {"name": "Makasar", "latitude": 0.9039, "longitude": 122.7586},
    {"name": "Padang", "latitude": -0.9492, "longitude": 100.3543},
    {"name": "Wonosobo", "latitude": -7.3589, "longitude": 109.9031},
    {"name": "Tegal", "latitude": -6.8694, "longitude": 109.1402},
    {"name": "Medan", "latitude": 3.5833, "longitude": 98.6667},
    {"name": "Palembang", "latitude": -2.9167, "longitude": 104.7458},
    {"name": "Bekasi", "latitude": -6.2349, "longitude": 106.9896},
    {"name": "Semarang", "latitude": -6.9931, "longitude": 110.4208},
    {"name": "Jogja", "latitude": -3.4292, "longitude": 119.222}
]

all_hourly_dataframes = {}

for location in locations:
    print(f"\nFetching data for {location['name']}...")
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": location['latitude'],
        "longitude": location['longitude'],
        "hourly": ["us_aqi", "us_aqi_pm2_5", "us_aqi_nitrogen_dioxide", "us_aqi_carbon_monoxide", "us_aqi_ozone"],
        "current": ["us_aqi", "nitrogen_dioxide", "ozone", "pm2_5", "carbon_monoxide"],
        "timezone": "Asia/Bangkok",
        "start_date": "2025-10-01",
        "end_date": "2025-10-31",
    }
    responses = openmeteo.weather_api(url, params=params)

    response = responses[0]
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

    current = response.Current()
    current_us_aqi = current.Variables(0).Value()
    current_nitrogen_dioxide = current.Variables(1).Value()
    current_ozone = current.Variables(2).Value()
    current_pm2_5 = current.Variables(3).Value()
    current_carbon_monoxide = current.Variables(4).Value()

    print(f"\nCurrent time: {current.Time()}")
    print(f"Current us_aqi: {current_us_aqi}")
    print(f"Current nitrogen_dioxide: {current_nitrogen_dioxide}")
    print(f"Current ozone: {current_ozone}")
    print(f"Current pm2_5: {current_pm2_5}")
    print(f"Current carbon_monoxide: {current_carbon_monoxide}")

    hourly = response.Hourly()
    hourly_us_aqi = hourly.Variables(0).ValuesAsNumpy()
    hourly_us_aqi_pm2_5 = hourly.Variables(1).ValuesAsNumpy()
    hourly_us_aqi_nitrogen_dioxide = hourly.Variables(2).ValuesAsNumpy()
    hourly_us_aqi_carbon_monoxide = hourly.Variables(3).ValuesAsNumpy()
    hourly_us_aqi_ozone = hourly.Variables(4).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
            end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
            freq = pd.Timedelta(seconds = hourly.Interval()),
            inclusive = "left"
        )
    }

    hourly_data["us_aqi"] = hourly_us_aqi
    hourly_data["us_aqi_pm2_5"] = hourly_us_aqi_pm2_5
    hourly_data["us_aqi_nitrogen_dioxide"] = hourly_us_aqi_nitrogen_dioxide
    hourly_data["us_aqi_carbon_monoxide"] = hourly_us_aqi_carbon_monoxide
    hourly_data["us_aqi_ozone"] = hourly_us_aqi_ozone

    hourly_dataframe = pd.DataFrame(data = hourly_data)
    all_hourly_dataframes[location['name']] = hourly_dataframe
    print(f"\nHourly data for {location['name']}:\n", hourly_dataframe.head())

print("\nAll hourly dataframes by location:")
for name, df in all_hourly_dataframes.items():
    print(f"\n{name} Hourly Data (first 5 rows):\n")
    display(df.head())



Fetching data for Bandung...
Coordinates: -6.900001525878906°N 107.60000610351562°E
Elevation: 707.0 m asl
Timezone: b'Asia/Bangkok'b'GMT+7'
Timezone difference to GMT+0: 25200s

Current time: 1762516800
Current us_aqi: 163.70394897460938
Current nitrogen_dioxide: 101.19999694824219
Current ozone: 9.0
Current pm2_5: 105.0
Current carbon_monoxide: 2882.0

Hourly data for Bandung:
                        date      us_aqi  us_aqi_pm2_5  \
0 2025-09-30 17:00:00+00:00  156.982452    156.982452   
1 2025-09-30 18:00:00+00:00  156.583328    156.583328   
2 2025-09-30 19:00:00+00:00  156.256577    156.256577   
3 2025-09-30 20:00:00+00:00  156.037277    156.037277   
4 2025-09-30 21:00:00+00:00  155.934219    155.934219   

   us_aqi_nitrogen_dioxide  us_aqi_carbon_monoxide  us_aqi_ozone  
0                37.775806               17.664251     22.147495  
1                35.657997               18.539856     14.262523  
2                33.392437               18.659420      8.000928  
3    

,date,us_aqi,us_aqi_pm2_5,us_aqi_nitrogen_dioxide,us_aqi_carbon_monoxide,us_aqi_ozone
0,2025-09-30 17:00:00+00:00,156.982452,156.982452,37.775806,17.664251,22.147495
1,2025-09-30 18:00:00+00:00,156.583328,156.583328,35.657997,18.539856,14.262523
2,2025-09-30 19:00:00+00:00,156.256577,156.256577,33.392437,18.659420,8.000928
3,2025-09-30 20:00:00+00:00,156.037277,156.037277,31.028370,17.452898,3.942486
4,2025-09-30 21:00:00+00:00,155.934219,155.934219,29.353821,15.097827,2.087198



Jakarta Hourly Data (first 5 rows):



,date,us_aqi,us_aqi_pm2_5,us_aqi_nitrogen_dioxide,us_aqi_carbon_monoxide,us_aqi_ozone
0,2025-09-30 17:00:00+00:00,165.892548,165.892548,57.735897,51.376091,39.598797
1,2025-09-30 18:00:00+00:00,166.721497,166.721497,54.960678,55.872829,24.466606
2,2025-09-30 19:00:00+00:00,167.517548,167.517548,52.474560,58.473915,12.697124
3,2025-09-30 20:00:00+00:00,168.348694,168.348694,50.219707,59.361958,5.623840
4,2025-09-30 21:00:00+00:00,169.118423,169.118423,49.349880,58.742393,3.304731



Surabaya Hourly Data (first 5 rows):



...

,date,us_aqi,us_aqi_pm2_5,us_aqi_nitrogen_dioxide,us_aqi_carbon_monoxide,us_aqi_ozone
0,2025-09-30 17:00:00+00:00,165.892548,165.892548,57.735897,31.519327,39.598797
1,2025-09-30 18:00:00+00:00,166.721497,166.721497,54.960678,36.198067,24.466606
2,2025-09-30 19:00:00+00:00,167.517548,167.517548,52.474560,39.978260,12.697124
3,2025-09-30 20:00:00+00:00,168.348694,168.348694,50.219707,42.733089,5.623840
4,2025-09-30 21:00:00+00:00,169.118423,169.118423,49.349880,44.265701,3.304731



Semarang Hourly Data (first 5 rows):



,date,us_aqi,us_aqi_pm2_5,us_aqi_nitrogen_dioxide,us_aqi_carbon_monoxide,us_aqi_ozone
0,2025-09-30 17:00:00+00:00,121.354172,121.354172,11.524823,9.566424,48.063541
1,2025-09-30 18:00:00+00:00,122.750008,122.750008,11.771079,10.031402,42.381725
2,2025-09-30 19:00:00+00:00,124.260414,124.260414,11.869582,10.342995,37.047775
3,2025-09-30 20:00:00+00:00,125.937508,125.937508,11.771079,10.351449,32.583488
4,2025-09-30 21:00:00+00:00,127.468765,127.468765,11.574074,10.047101,27.887291



Jogja Hourly Data (first 5 rows):



,date,us_aqi,us_aqi_pm2_5,us_aqi_nitrogen_dioxide,us_aqi_carbon_monoxide,us_aqi_ozone
0,2025-09-30 17:00:00+00:00,47.274303,47.274303,2.068558,3.429951,29.568645
1,2025-09-30 18:00:00+00:00,47.274303,47.274303,2.167061,3.613527,28.525045
2,2025-09-30 19:00:00+00:00,47.204861,47.204861,2.314815,3.654589,27.655380
3,2025-09-30 20:00:00+00:00,47.291676,47.291676,2.462569,3.572464,27.017624
4,2025-09-30 21:00:00+00:00,47.395836,47.395836,2.413318,3.394928,26.611782


In [21]:
total_data_points = 0
for location_name, df in all_hourly_dataframes.items():
    print(f"Jumlah data per jam untuk {location_name}: {len(df)} baris")
    total_data_points += len(df)

print(f"\nTotal semua data per jam dari semua lokasi: {total_data_points} baris")

Jumlah data per jam untuk Bandung: 744 baris
Jumlah data per jam untuk Jakarta: 744 baris
Jumlah data per jam untuk Surabaya: 744 baris
Jumlah data per jam untuk Bali: 744 baris
Jumlah data per jam untuk Pekanbaru: 744 baris
Jumlah data per jam untuk Makasar: 744 baris
Jumlah data per jam untuk Padang: 744 baris
Jumlah data per jam untuk Wonosobo: 744 baris
Jumlah data per jam untuk Tegal: 744 baris
Jumlah data per jam untuk Medan: 744 baris
Jumlah data per jam untuk Palembang: 744 baris
Jumlah data per jam untuk Bekasi: 744 baris
Jumlah data per jam untuk Semarang: 744 baris
Jumlah data per jam untuk Jogja: 744 baris

Total semua data per jam dari semua lokasi: 10416 baris


In [22]:
def get_aqi_category(value, pollutant_type):
    if pollutant_type == 'AQI':
        if 0 <= value <= 50: return 'Good'
        elif 51 <= value <= 100: return 'Moderate'
        elif 101 <= value <= 150: return 'Unhealthy for Sensitive Groups'
        elif 151 <= value <= 200: return 'Unhealthy'
        elif 201 <= value <= 300: return 'Very Unhealthy'
        else: return 'Hazardous'
    elif pollutant_type == 'CO':
        if 0 <= value <= 4.5: return 'Good'
        elif 4.6 <= value <= 9.5: return 'Moderate'
        elif 9.6 <= value <= 12.5: return 'Unhealthy for Sensitive Groups'
        elif 12.6 <= value <= 15.5: return 'Unhealthy'
        elif 15.6 <= value <= 30.5: return 'Very Unhealthy'
        else: return 'Hazardous'
    elif pollutant_type == 'Ozone':
        if 0 <= value <= 50: return 'Good'
        elif 51 <= value <= 125: return 'Moderate'
        elif 126 <= value <= 165: return 'Unhealthy for Sensitive Groups'
        elif 166 <= value <= 205: return 'Unhealthy'
        elif 206 <= value <= 405: return 'Very Unhealthy'
        else: return 'Hazardous'
    elif pollutant_type == 'NO2':
        if 0 <= value <= 54: return 'Good'
        elif 55 <= value <= 100: return 'Moderate'
        elif 101 <= value <= 360: return 'Unhealthy for Sensitive Groups'
        elif 361 <= value <= 650: return 'Unhealthy'
        elif 651 <= value <= 1250: return 'Very Unhealthy'
        else: return 'Hazardous'
    elif pollutant_type == 'PM2.5':
        if 0 <= value <= 12: return 'Good'
        elif 13 <= value <= 35.5: return 'Moderate'
        elif 35.6 <= value <= 55.5: return 'Unhealthy for Sensitive Groups'
        elif 55.6 <= value <= 150.5: return 'Unhealthy'
        elif 150.6 <= value <= 250.5: return 'Very Unhealthy'
        else: return 'Hazardous'
    return 'Unknown'

processed_dfs = []

for city_name, df in all_hourly_dataframes.items():
    temp_df = df.copy()
    temp_df['Country'] = 'Indonesia'
    temp_df['City'] = city_name

    temp_df = temp_df.rename(columns={
        'us_aqi': 'AQI Value',
        'us_aqi_carbon_monoxide': 'CO AQI Value',
        'us_aqi_ozone': 'Ozone AQI Value',
        'us_aqi_nitrogen_dioxide': 'NO2 AQI Value',
        'us_aqi_pm2_5': 'PM2.5 AQI Value'
    })

    temp_df['AQI Category'] = temp_df['AQI Value'].apply(lambda x: get_aqi_category(x, 'AQI'))
    temp_df['CO AQI Category'] = temp_df['CO AQI Value'].apply(lambda x: get_aqi_category(x, 'CO'))
    temp_df['Ozone AQI Category'] = temp_df['Ozone AQI Value'].apply(lambda x: get_aqi_category(x, 'Ozone'))
    temp_df['NO2 AQI Category'] = temp_df['NO2 AQI Value'].apply(lambda x: get_aqi_category(x, 'NO2'))
    temp_df['PM2.5 AQI Category'] = temp_df['PM2.5 AQI Value'].apply(lambda x: get_aqi_category(x, 'PM2.5'))

    final_columns = [
        'date', 'Country', 'City',
        'AQI Value', 'AQI Category',
        'CO AQI Value', 'CO AQI Category',
        'Ozone AQI Value', 'Ozone AQI Category',
        'NO2 AQI Value', 'NO2 AQI Category',
        'PM2.5 AQI Value', 'PM2.5 AQI Category'
    ]
    temp_df = temp_df[final_columns]

    processed_dfs.append(temp_df)

combined_air_quality_df = pd.concat(processed_dfs, ignore_index=True)
combined_air_quality_df = combined_air_quality_df.drop(columns=['date'])

print("Combined Air Quality DataFrame (first 5 rows):")
display(combined_air_quality_df.head())

print("\nDataFrame Info:")
combined_air_quality_df.info()


Combined Air Quality DataFrame (first 5 rows):


,Country,City,AQI Value,AQI Category,CO AQI Value,CO AQI Category,Ozone AQI Value,Ozone AQI Category,NO2 AQI Value,NO2 AQI Category,PM2.5 AQI Value,PM2.5 AQI Category
0,Indonesia,Bandung,156.982452,Unhealthy,17.664251,Very Unhealthy,22.147495,Good,37.775806,Good,156.982452,Very Unhealthy
1,Indonesia,Bandung,156.583328,Unhealthy,18.539856,Very Unhealthy,14.262523,Good,35.657997,Good,156.583328,Very Unhealthy
2,Indonesia,Bandung,156.256577,Unhealthy,18.659420,Very Unhealthy,8.000928,Good,33.392437,Good,156.256577,Very Unhealthy
3,Indonesia,Bandung,156.037277,Unhealthy,17.452898,Very Unhealthy,3.942486,Good,31.028370,Good,156.037277,Very Unhealthy
4,Indonesia,Bandung,155.934219,Unhealthy,15.097827,Unhealthy,2.087198,Good,29.353821,Good,155.934219,Very Unhealthy



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10416 entries, 0 to 10415
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country             10416 non-null  object 
 1   City                10416 non-null  object 
 2   AQI Value           10416 non-null  float32
 3   AQI Category        10416 non-null  object 
 4   CO AQI Value        10416 non-null  float32
 5   CO AQI Category     10416 non-null  object 
 6   Ozone AQI Value     10416 non-null  float32
 7   Ozone AQI Category  10416 non-null  object 
 8   NO2 AQI Value       10416 non-null  float32
 9   NO2 AQI Category    10416 non-null  object 
 10  PM2.5 AQI Value     10416 non-null  float32
 11  PM2.5 AQI Category  10416 non-null  object 
dtypes: float32(5), object(7)
memory usage: 773.2+ KB


In [24]:
combined_air_quality_df.to_csv('air_quality_data(fetch).csv', index=False)
print("Dataset telah berhasil disimpan")

Dataset telah berhasil disimpan
